In [ ]:
# ====================================

# Heart Disease Detection – Next Stage

# Using HRV features extracted from ECG images

# ====================================
 
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns
 
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
 
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
 
import shap
 
# 1️⃣ Load the cleaned dataset (you saved this as "heart_hrv_cleaned.csv")

df = pd.read_csv("heart_hrv_cleaned.csv")
 
# 2️⃣ Prepare X and y

X = df.drop(columns=["label", "image"], errors='ignore')

y = df["label"]
 
# 3️⃣ Encode y

le = LabelEncoder()

y_enc = le.fit_transform(y)

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
 
# 4️⃣ Split into train/test

X_train, X_test, y_train, y_test = train_test_split(

    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42

)
 
# 5️⃣ Feature preprocessing: handle missing/inf, scaling

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[

    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),

    ('scaler', StandardScaler())

])
 
preprocessor = ColumnTransformer(

    transformers=[

        ('num', numeric_transformer, numeric_features)

    ],

    remainder='passthrough'

)
 
# Fit transform the train and transform test

X_train_pp = preprocessor.fit_transform(X_train)

X_test_pp = preprocessor.transform(X_test)
 
# 6️⃣ Feature importance: Random Forest

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

rf.fit(X_train_pp, y_train)
 
importances = rf.feature_importances_

feat_imp = pd.Series(importances, index=numeric_features).sort_values(ascending=False)

print("Top features:\n", feat_imp.head(20))
 
# Plot

plt.figure(figsize=(10,8))

feat_imp.head(20).plot(kind='barh')

plt.title("Top 20 Feature Importances (Random Forest)")

plt.gca().invert_yaxis()

plt.show()
 
# 7️⃣ Feature selection: choose top N features (e.g., 20 or 30)

topN = 30

top_features = feat_imp.head(topN).index.tolist()

print("Selected top features:", top_features)
 
# Reduce datasets

X_train_sel = pd.DataFrame(X_train_pp, columns=numeric_features)[top_features].values

X_test_sel = pd.DataFrame(X_test_pp, columns=numeric_features)[top_features].values
 
# 8️⃣ Model building: Ensemble of XGBoost + RF + Logistic Regression

xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, use_label_encoder=False, eval_metric='mlogloss', random_state=42)

rf2 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

lr = LogisticRegression(max_iter=500, random_state=42)
 
voting = VotingClassifier(

    estimators=[('xgb', xgb), ('rf', rf2), ('lr', lr)],

    voting='soft',

    n_jobs=-1

)
 
# Fit

voting.fit(X_train_sel, y_train)
 
# Predict & evaluate

y_pred = voting.predict(X_test_sel)

y_prob = voting.predict_proba(X_test_sel)
 
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))
 
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()
 
# 9️⃣ Prediction probabilities: show for a few samples

prob_df = pd.DataFrame(y_prob, columns=le.classes_)

print("\nSample probabilities:\n", prob_df.head())
 
#  🔟 Explainability: SHAP for XGBoost (or the ensemble if you wrap explainer accordingly)

explainer = shap.TreeExplainer(xgb)

shap_values = explainer.shap_values(X_test_sel)

shap.summary_plot(shap_values, pd.DataFrame(X_test_sel, columns=top_features))
 
# 1️⃣1️⃣ Save your model and feature list for deployment

import joblib

joblib.dump(voting, "heart_disease_ensemble_model.pkl")

joblib.dump(top_features, "top_features_list.pkl")

joblib.dump(le, "label_encoder.pkl")
 
print("Models & feature list saved.")
 
# 1️⃣2️⃣ (Optional) If you want to retrain or tune further:

# Use GridSearchCV or RandomizedSearchCV on XGBoost or RF etc.

 

In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns
 
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
 
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
 
import shap